# MAP 1 - Funciones Vectoriales - MK8 Mario Circuit  - Matemática 4 (308)

# 0. Introducción.

En este MAP, nos enfocaremos en utilizar las *funciones vectoriales* para modelar la consturcción de pistas.En esta segunda parte trabajaremos directamente *a partir de las funciones vectoriales* que se obtuvieron en la parte 1 y en un modelo alternativo generado por la IA.

### Objetivos
1. Definir múltiples funciones vectoriales (tramos) y sus intervalos.
2. Generar puntos desde las funciones y unirlos en un único modelo.
3. Visualizar la curva en 3D.
4. Realizar análisis físico: velocidad, aceleración, componentes tangencial y normal, curvatura, planos osculador y normal, circunferencia osculatriz.
5. Simular movimiento sobre la curva (animación interactiva) y mostrar gráficas de velocidad y aceleración.
6. Sugerir mejoras de precisión si se proporcionan datos reales.

**Nota:** Este notebook contiene celdas con funciones vacías para que puedan completarlas.

## Recomendaciones de Liberias
- Python 3.10+.
- Librerías: `numpy`, `pandas`, `matplotlib`, `plotly`, `scipy`, `sympy`.
- Opcional: `opencv-python` (solo si desean procesar imágenes).

Instalación (si es necesario):

```bash
pip install numpy pandas matplotlib plotly scipy sympy opencv-python
```

Se recomienda abrir el notebook en JupyterLab/Jupyter Notebook o en Google Colab.

In [ ]:
# === Librerías comunes ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy.signal import savgol_filter
import sympy as sp
from IPython.display import display
plt.style.use('seaborn-v0_8')


# Parte 2 - Explorando a Fondo

Utilice **Google Colab** u otro entorno similar (consulte la Sección 5) para trabajar con el Notebook de Python adjunto en la asignación MAP 1, disponible en el portal GES del curso. Emplee cualquier herramienta de **inteligencia artificial (IA)** (ver Sección 5) para modificar y complementar dicho Notebook. Tome en cuenta que, cada consulta a la IA debe ser **documentada** en su reporte con un **screenshot** del prompt y la respuesta obtenida, explicando cómo se utilizó. En el Notebook, desarrolle cada uno de los puntos que se detallan a continuación.

$1.$ Agregue el código de su modelo matemático implementado en la Parte 1. Consulte a la IA sobre posibles mejoras y documente el resultado.

In [ ]:
############################ AGREGUE SU IMPLEMENTACIÓN EN CÓDIGO EN ESTA PARTE ###########################

t = sp.symbols('t', real=True)

#Documento modificado de modelo matemático generado por CalcPlot3D. 
#Utilicé Claude para convertir el modelo de CalcPlot a uno que pueda usar nuestras herramientas
curvas_def = {
    'r1': {
        'x': 25 * sp.cos(t),
        'y': 25 * sp.sin(t) + 25,
        'z': (10 / sp.pi) * t,
        'tmin': 0,
        'tmax': sp.pi,
        'tSteps': 500,
        'color': 'red',
    },
    'r2': {
        'x': sp.Integer(-25),
        'y': t,
        'z': 0.01 * t**2 - 0.0055556 * t + 3.8889,
        'tmin': -20,
        'tmax': 25,
        'tSteps': 500,
        'color': 'blue',
    },
    'r3': {
        'x': -25 * sp.cos(t),
        'y': -25 * sp.sin(t) - 20,
        'z': -(12 / (7 * sp.pi)) * t + 8,
        'tmin': 0,
        'tmax': sp.Rational(7, 6) * sp.pi,
        'tSteps': 800,
        'color': 'darkgreen',
    },
    'r4': {
        'x': 0.04 * t**2 + 0.091429 * t + 20.00857,
        'y': t,
        'z': sp.Integer(6),
        'tmin': -7.50,
        'tmax': 10,
        'tSteps': 400,
        'color': 'darkred',
    },
    'r5': {
        'x': sp.Integer(25),
        'y': 10 + 15 * t,
        'z': 6 - 6 * t,
        'tmin': 0,
        'tmax': 1,
        'tSteps': 200,
        'color': 'darkblue',
    },
}




############################################

$2.$ Grafique  en Python su modelo matemático con una animación que simule el movimiento del carro a lo largo de la pista.

In [ ]:
############################ AGREGUE SU IMPLEMENTACIÓN EN CÓDIGO EN ESTA PARTE ###########################

# Preparación de la curva
class CurvaParametrica:
    """Modelo r(t) = (x(t), y(t), z(t)) con cinemática asociada."""

    def __init__(self, nombre, x_expr, y_expr, z_expr, tmin, tmax, tSteps, color):
        self.nombre = nombre
        self.color = color
        self.tmin = float(tmin)
        self.tmax = float(tmax)
        self.tSteps = tSteps

        # Vector posición simbólico
        self.r = sp.Matrix([x_expr, y_expr, z_expr])

        # Derivadas simbólicas: velocidad y aceleración
        self.v = self.r.diff(t)          # r'(t)  -> velocidad
        self.a = self.v.diff(t)          # r''(t) -> aceleración

        # Rapidez |r'(t)|
        self.rapidez = sp.sqrt(sum(comp**2 for comp in self.v))

        # Curvatura kappa(t) = |r' x r''| / |r'|^3  (fórmula estándar)
        cross = self.v.cross(self.a)
        norm_cross = sp.sqrt(sum(c**2 for c in cross))
        self.curvatura = norm_cross / self.rapidez**3

        # Funciones numéricas rápidas (numpy) a partir de las expresiones
        self._fx = sp.lambdify(t, x_expr, 'numpy')
        self._fy = sp.lambdify(t, y_expr, 'numpy')
        self._fz = sp.lambdify(t, z_expr, 'numpy')
        self._fvx = sp.lambdify(t, self.v[0], 'numpy')
        self._fvy = sp.lambdify(t, self.v[1], 'numpy')
        self._fvz = sp.lambdify(t, self.v[2], 'numpy')
        self._frapidez = sp.lambdify(t, self.rapidez, 'numpy')
        self._fcurvatura = sp.lambdify(t, self.curvatura, 'numpy')

    def evaluar(self):
        """Devuelve un DataFrame con posición, velocidad, rapidez y curvatura."""
        ts = np.linspace(self.tmin, self.tmax, self.tSteps)

        x = np.full_like(ts, self._fx(ts)) if np.isscalar(self._fx(ts)) else self._fx(ts)
        y = np.full_like(ts, self._fy(ts)) if np.isscalar(self._fy(ts)) else self._fy(ts)
        z = np.full_like(ts, self._fz(ts)) if np.isscalar(self._fz(ts)) else self._fz(ts)

        vx = np.full_like(ts, self._fvx(ts)) if np.isscalar(self._fvx(ts)) else self._fvx(ts)
        vy = np.full_like(ts, self._fvy(ts)) if np.isscalar(self._fvy(ts)) else self._fvy(ts)
        vz = np.full_like(ts, self._fvz(ts)) if np.isscalar(self._fvz(ts)) else self._fvz(ts)

        rapidez = self._frapidez(ts)
        rapidez = np.full_like(ts, rapidez) if np.isscalar(rapidez) else rapidez

        try:
            curvatura = self._fcurvatura(ts)
            curvatura = np.full_like(ts, curvatura) if np.isscalar(curvatura) else curvatura
        except ZeroDivisionError:
            curvatura = np.full_like(ts, np.nan)

        df = pd.DataFrame({
            't': ts,
            'x': x, 'y': y, 'z': z,
            'vx': vx, 'vy': vy, 'vz': vz,
            'rapidez': rapidez,
            'curvatura': curvatura,
            'curva': self.nombre,
        })
        return df

    def resumen_simbolico(self):
        print(f"--- {self.nombre} ---")
        print("r(t)  =", tuple(self.r))
        print("r'(t) =", tuple(self.v))
        print("r''(t)=", tuple(self.a))
        print("|r'(t)| =", sp.simplify(self.rapidez))
        print()

    def longitud_arco(self):
        """
        Longitud de arco:  L = ∫_{tmin}^{tmax} |r'(t)| dt

        Primero intenta resolver la integral de forma simbólica (exacta) con
        sympy; si la expresión no tiene antiderivada elemental o sympy no
        logra evaluarla en el intervalo, recurre a integración numérica
        (cuadratura adaptativa) con scipy.integrate.quad.
        Devuelve (L, exacta: bool).
        """
        # --- Intento simbólico ---
        try:
            L_sym = sp.integrate(self.rapidez, (t, self.tmin, self.tmax))
            L_val = float(sp.N(L_sym))
            if np.isfinite(L_val):
                return L_val, True
        except Exception:
            pass

        # --- Respaldo numérico (cuadratura adaptativa) ---
        f_rapidez = sp.lambdify(t, self.rapidez, 'numpy')
        L_num, _error_est = quad(f_rapidez, self.tmin, self.tmax, limit=200)
        return L_num, False

curvas = {}
dataframes = []

for nombre, d in curvas_def.items():
    c = CurvaParametrica(nombre, d['x'], d['y'], d['z'],
                          d['tmin'], d['tmax'], d['tSteps'], d['color'])
    curvas[nombre] = c
    c.resumen_simbolico()
    dataframes.append(c.evaluar())

df_total = pd.concat(dataframes, ignore_index=True)

# Suavizado opcional de la rapidez por tramo (útil si se muestrean datos ruidosos)
df_total['rapidez_suave'] = df_total.groupby('curva')['rapidez'].transform(
    lambda s: savgol_filter(s, window_length=min(51, len(s) - (1 - len(s) % 2)), polyorder=3)
    if len(s) > 51 else s
)

# Animación final

orden_recorrido = [
    ('r1', False),  # False = usar t de tmin a tmax (orden natural)
    ('r2', True),   # True  = usar t de tmax a tmin (invertido)
    ('r3', False),
    ('r4', False),
    ('r5', False),
]

tramos_ordenados = []
for nombre, invertir in orden_recorrido:
    d = df_total[df_total['curva'] == nombre].copy()
    if invertir:
        d = d.iloc[::-1].reset_index(drop=True)
    tramos_ordenados.append(d)

df_recorrido = pd.concat(tramos_ordenados, ignore_index=True)

# Down-sample a un número manejable de cuadros para la animación
N_FRAMES = 300
idx_frames = np.linspace(0, len(df_recorrido) - 1, N_FRAMES).astype(int)
df_anim = df_recorrido.iloc[idx_frames].reset_index(drop=True)

# ---------------------------------------------------------------------------
# 6a) Animación con matplotlib (se exporta como GIF)
# ---------------------------------------------------------------------------
from matplotlib.animation import FuncAnimation, PillowWriter

fig3 = plt.figure(figsize=(9, 7))
ax3 = fig3.add_subplot(111, projection='3d')

# Trayectoria completa de fondo (tenue), coloreada por tramo original
for nombre, c in curvas.items():
    d = df_total[df_total['curva'] == nombre]
    ax3.plot(d['x'], d['y'], d['z'], color=c.color, linewidth=1.5, alpha=0.5, label=nombre)

ax3.set_xlabel('x')
ax3.set_ylabel('y')
ax3.set_zlabel('z')
ax3.set_title('Simulación del movimiento del punto (sentido antihorario, inicio en r1)')
ax3.legend(loc='upper left', fontsize=8)
ax3.set_xlim(df_recorrido['x'].min() - 5, df_recorrido['x'].max() + 5)
ax3.set_ylim(df_recorrido['y'].min() - 5, df_recorrido['y'].max() + 5)
ax3.set_zlim(df_recorrido['z'].min() - 2, df_recorrido['z'].max() + 2)

# Estela (rastro recorrido) y punto móvil
estela, = ax3.plot([], [], [], color='black', linewidth=2.5)
punto, = ax3.plot([], [], [], marker='o', markersize=8, color='crimson')
texto = ax3.text2D(0.02, 0.95, '', transform=ax3.transAxes, fontsize=9)


def init_anim():
    estela.set_data([], [])
    estela.set_3d_properties([])
    punto.set_data([], [])
    punto.set_3d_properties([])
    texto.set_text('')
    return estela, punto, texto


def update_anim(frame_i):
    xs = df_anim['x'].iloc[:frame_i + 1]
    ys = df_anim['y'].iloc[:frame_i + 1]
    zs = df_anim['z'].iloc[:frame_i + 1]
    estela.set_data(xs, ys)
    estela.set_3d_properties(zs)

    x_pt = df_anim['x'].iloc[frame_i]
    y_pt = df_anim['y'].iloc[frame_i]
    z_pt = df_anim['z'].iloc[frame_i]
    punto.set_data([x_pt], [y_pt])
    punto.set_3d_properties([z_pt])

    curva_actual = df_anim['curva'].iloc[frame_i]
    texto.set_text(f"Tramo actual: {curva_actual}")
    return estela, punto, texto


anim = FuncAnimation(
    fig3, update_anim, frames=len(df_anim),
    init_func=init_anim, interval=40, blit=False, repeat=True,
)

anim.save('c:\\Users\\nelel\\Downloads\\MAP1_Funciones_Vectoriales\\simulacion_punto.gif', writer=PillowWriter(fps=25))
plt.close(fig3)


############################################

--- r1 ---
r(t)  = (25*cos(t), 25*sin(t) + 25, 10*t/pi)
r'(t) = (-25*sin(t), 25*cos(t), 10/pi)
r''(t)= (-25*cos(t), -25*sin(t), 0)
|r'(t)| = 5*sqrt(4 + 25*pi**2)/pi

--- r2 ---
r(t)  = (-25, t, 0.01*t**2 - 0.0055556*t + 3.8889)
r'(t) = (0, 1, 0.02*t - 0.0055556)
r''(t)= (0, 0, 0.0200000000000000)
|r'(t)| = sqrt((0.02*t - 0.0055556)**2 + 1)

--- r3 ---
r(t)  = (-25*cos(t), -25*sin(t) - 20, -12*t/(7*pi) + 8)
r'(t) = (25*sin(t), -25*cos(t), -12/(7*pi))
r''(t)= (25*cos(t), 25*sin(t), 0)
|r'(t)| = sqrt(144 + 30625*pi**2)/(7*pi)

--- r4 ---
r(t)  = (0.04*t**2 + 0.091429*t + 20.00857, t, 6)
r'(t) = (0.08*t + 0.091429, 1, 0)
r''(t)= (0.0800000000000000, 0, 0)
|r'(t)| = sqrt((0.08*t + 0.091429)**2 + 1)

--- r5 ---
r(t)  = (25, 15*t + 10, 6 - 6*t)
r'(t) = (0, 15, -6)
r''(t)= (0, 0, 0)
|r'(t)| = 3*sqrt(29)



$3.$ Verifique la longitud de arco de la pista obtenida en la Parte 1.

In [ ]:
############################ AGREGUE SU IMPLEMENTACIÓN EN CÓDIGO EN ESTA PARTE ###########################


longitudes = []
longitudTotal = 0
for nombre, c in curvas.items():
    L, es_exacta = c.longitud_arco()
    longitudTotal += L
    longitudes.append({
        'curva': nombre,
        'tmin': c.tmin,
        'tmax': c.tmax,
        'longitud_arco': L,
        'metodo': 'simbólico (exacto)' if es_exacta else 'numérico (quad)',
    })

df_longitudes = pd.DataFrame(longitudes)
df_longitudes.loc['TOTAL'] = {
    'curva': 'Lazo completo (r1+r2+r3+r4+r5)',
    'tmin': np.nan, 'tmax': np.nan,
    'longitud_arco': df_longitudes['longitud_arco'].sum(),
    'metodo': 'suma de tramos',
}

print("=== Longitud de arco por tramo ===")
print(df_longitudes.to_string(index=False))
print(L)
print()


############################################

$4.$ Suponga que un vehículo recorre la pista con una rapidez constante de $35 \,\text{m/s}$. Consulte a la IA el punto de mayor curvatura $\kappa$ de la pista. ¿Qué consideraciones deben tenerse en la construcción de la pista? ¿Qué ocurre si la rapidez de los vehículos no es constante?

In [ ]:
############################ AGREGUE SU IMPLEMENTACIÓN EN CÓDIGO EN ESTA PARTE ###########################


candidatos = []
for nombre, c in curvas.items():
    d = df_total[df_total['curva'] == nombre].reset_index(drop=True)
    d_interior = d.iloc[1:-1]  # se excluyen primer y último punto muestreado
    fila_max = d_interior.loc[d_interior['curvatura'].idxmax()]
    candidatos.append(fila_max)
 
df_candidatos = pd.DataFrame(candidatos)
fila_ganadora = df_candidatos.loc[df_candidatos['curvatura'].idxmax()]
 
print("=== Curvatura máxima por tramo (sin contar empalmes) ===")
print(df_candidatos[['curva', 't', 'x', 'y', 'z', 'curvatura']].to_string(index=False))
print()
print(f">>> Punto de mayor curvatura en toda la pista: tramo {fila_ganadora['curva']}, "
      f"t = {fila_ganadora['t']:.4f}")
print(f"    Coordenadas: ({fila_ganadora['x']:.4f}, {fila_ganadora['y']:.4f}, {fila_ganadora['z']:.4f})")
print(f"    Curvatura kappa = {fila_ganadora['curvatura']:.6f}")
print()


############################################

$5.$ Usando la imagen de la pista, utilice IA para construir un modelo matemático alternativo de la trayectoria (por ejemplo, mediante curvas tipo Lissajous) y grafíquelo en Python considerando la animación del carro a lo largo de la pista.

In [ ]:
############################ AGREGUE SU IMPLEMENTACIÓN EN CÓDIGO EN ESTA PARTE ###########################





############################################

$6.$ Compare los dos modelos: el suyo y el alternativo generado por la IA en el inciso anterior. Discuta las ventajas y desventajas de cada uno.

In [ ]:
############################ AGREGUE SU IMPLEMENTACIÓN EN CÓDIGO EN ESTA PARTE ###########################





############################################

$7.$ Con ayuda de la IA y ambos modelos matemáticos, responda:

- ¿Cómo comprobar en Python que la longitud de arco es coherente con la escala de la pista en ambos modelos?

In [ ]:
########################################################################################################
#  ####### AGREGUE SU IMPLEMENTACIÓN  EN CÓDIGO PARA JUSTIFICAR SU RESPUESTA EN ESTA PARTE #############
# ##############





############################################

- ¿Qué aplicación tiene el esquema TNB en este problema?

In [ ]:
########################################################################################################
#  ####### AGREGUE SU IMPLEMENTACIÓN  EN CÓDIGO PARA JUSTIFICAR SU RESPUESTA EN ESTA PARTE #############
# ##############





############################################

- ¿Cómo interpretaría la curvatura máxima en términos de dificultad de conducción?

In [ ]:
########################################################################################################
#  ####### AGREGUE SU IMPLEMENTACIÓN  EN CÓDIGO PARA JUSTIFICAR SU RESPUESTA EN ESTA PARTE #############
# ##############





############################################

- ¿Cómo modificaría $z(t)$ en ambos modelos de la pista para simular un puente más pronunciado o un túnel más largo?

In [ ]:
########################################################################################################
#  ####### AGREGUE SU IMPLEMENTACIÓN  EN CÓDIGO PARA JUSTIFICAR SU RESPUESTA EN ESTA PARTE #############
# ##############





############################################

- Encuentre la velocidad máxima para que el vehículo no se salga de la pista en los puntos marcados como $A$ y $B$ en la Figura 3.

In [ ]:
########################################################################################################
#  ####### AGREGUE SU IMPLEMENTACIÓN  EN CÓDIGO PARA JUSTIFICAR SU RESPUESTA EN ESTA PARTE #############
# ##############





############################################

**Nota:** *las respuestas a estas preguntas deben redactarse sin apoyo de herramientas de IA, utilizando únicamente su propio análisis y juicio crítico. Puede apoyarse con visualizaciones en Python.* 